# GolStats - Gold Analytics

This notebook transforms the cleaned Silver event data into analytical datasets
designed to answer football performance questions.

The Gold layer focuses on business and analytical metrics rather than raw data
transformation.

The main objectives are:

* Aggregate event-level data into team and player performance metrics.
* Calculate football-specific KPIs such as pass completion, shots, goals, and xG.
* Create datasets that can be consumed by analytical tools such as Power BI.
* Keep the resulting tables focused on specific analytical use cases.

Flow:

![image_1788525633710.png](./image_1788525633710.png "image_1788525633710.png")


## 1. Team match statistics

The first Gold dataset will provide team-level statistics for each match.

Each row will represent one team in one match.

The initial metrics will include:

* Total events
* Pass attempts
* Completed passes
* Pass completion percentage
* Shots
* Goals
* Expected goals (xG)
* Pressures
* Ball recoveries
* Dispossessions

Keeping the match as part of the analytical grain allows teams to be compared
within individual matches while also supporting aggregated tournament-level
analysis later.


In [0]:
from pyspark.sql.functions import col, count, sum, when, round

In [0]:
df_silver = spark.table("golstats.silver.eventos")

display(df_silver)

event_id,match_id,index,timestamp,period,minute,second,event_type,team_id,team,player_id,player,position,possession,possession_team_id,possession_team,x,y,play_pattern,under_pressure,counterpress,pass_length,pass_angle,pass_outcome,pass_recipient_id,pass_recipient,pass_technique,pass_height,pass_cross,pass_cut_back,pass_through_ball,pass_switch,pass_goal_assist,pass_end_x,pass_end_y,shot_xg,shot_outcome,shot_technique,shot_body_part,shot_first_time,shot_one_on_one,shot_end_x,shot_end_y
1f1c28b2-afc0-4986-a965-7e545df6941c,3857254,1,00:00:00.000,1,0,0,Starting XI,776,Denmark,null,null,null,1,776,Denmark,null,null,Regular Play,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
9de6bcb8-368e-40d7-91c2-fabb5b932caf,3857254,2,00:00:00.000,1,0,0,Starting XI,777,Tunisia,null,null,null,1,776,Denmark,null,null,Regular Play,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
8e0db96c-ed4c-410f-95f2-650f0de594bb,3857254,3,00:00:00.000,1,0,0,Half Start,776,Denmark,null,null,null,1,776,Denmark,null,null,Regular Play,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
6949d93f-9fca-434a-8d66-ef2dccc8dcd2,3857254,4,00:00:00.000,1,0,0,Half Start,777,Tunisia,null,null,null,1,776,Denmark,null,null,Regular Play,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
9266e43f-c76a-413b-8c25-9b61c1aefe49,3857254,5,00:00:00.361,1,0,0,Pass,776,Denmark,6302,Kasper Dolberg,Left Center Forward,2,776,Denmark,60.0,40.0,From Kick Off,null,null,3.9446166,-2.1025205,Complete,3043,Christian Dannemann Eriksen,null,Ground Pass,null,null,null,null,null,58.0,36.6,null,null,null,null,null,null,null,null
86eb15d6-28af-43b9-952d-c8fa4d56c6e4,3857254,6,00:00:00.580,1,0,0,Ball Receipt*,776,Denmark,3043,Christian Dannemann Eriksen,Left Center Midfield,2,776,Denmark,58.0,36.6,From Kick Off,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
ebf689ee-935c-43a6-a629-387a3a9db356,3857254,7,00:00:00.580,1,0,0,Carry,776,Denmark,3043,Christian Dannemann Eriksen,Left Center Midfield,2,776,Denmark,58.0,36.6,From Kick Off,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
e59abeb7-b8a4-49f8-9580-93b06348e5de,3857254,8,00:00:01.560,1,0,1,Pass,776,Denmark,3043,Christian Dannemann Eriksen,Left Center Midfield,2,776,Denmark,59.3,40.7,From Kick Off,null,null,19.36001,0.64246804,Complete,17042,Andreas Skov Olsen,null,Ground Pass,null,null,null,null,null,74.8,52.3,null,null,null,null,null,null,null,null
26bfda4b-25b7-4ac3-ae73-d41a11c49a4a,3857254,9,00:00:02.816,1,0,2,Ball Receipt*,776,Denmark,17042,Andreas Skov Olsen,Right Center Forward,2,776,Denmark,74.8,52.3,From Kick Off,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
103aeb5d-a816-4ed7-a12c-b56a3f61084a,3857254,10,00:00:02.816,1,0,2,Carry,776,Denmark,17042,Andreas Skov Olsen,Right Center Forward,2,776,Denmark,74.8,52.3,From Kick Off,true,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null


In [0]:
display(
    df_silver
    .filter(col("event_type") == "Shot")
    .groupBy("shot_outcome")
    .count()
    .orderBy("count", ascending=False)
)

shot_outcome,count
Off T,70
Blocked,58
Saved,36
Goal,23
Wayward,8
Post,4
Saved to Post,1


In [0]:
display(
    df_silver
    .filter(col("event_type") == "Shot")
    .select(
        "team",
        "player",
        "shot_xg",
        "shot_outcome"
    )
    .orderBy(col("shot_xg").desc())
    .limit(20)
)

team,player,shot_xg,shot_outcome
Wales,Gareth Frank Bale,0.7835,Goal
Canada,Alphonso Davies,0.7835,Saved
Argentina,Lionel Andrés Messi Cuccittini,0.7835,Goal
Ecuador,Enner Remberto Valencia Lastra,0.7835,Goal
Iran,Mehdi Taremi,0.7835,Goal
Poland,Robert Lewandowski,0.7835,Saved
France,Olivier Giroud,0.77927965,Goal
Argentina,Nicolás Alejandro Tagliafico,0.63619345,Saved
England,Jack Grealish,0.6262809,Goal
Denmark,Andreas Evald Cornelius,0.5683119,Post


## 2. Define team performance metrics

The team-level metrics are derived from the event types and attributes available
in the Silver dataset.

Each metric is explicitly defined to avoid ambiguity and ensure that the same
business logic can be reproduced consistently.

Metric definitions:

* `total_events`: total number of events associated with the team.
* `passes`: number of Pass events.
* `completed_passes`: Pass events classified as `Complete`.
* `pass_completion_pct`: completed passes divided by total passes.
* `shots`: number of Shot events.
* `goals`: Shot events with a `Goal` outcome.
* `xg`: sum of StatsBomb expected goals for the team's shots.
* `pressures`: number of Pressure events.
* `ball_recoveries`: number of Ball Recovery events.
* `dispossessions`: number of Dispossessed events.

The grain of the resulting dataset is:

One row = one team in one match.

This definition allows the metrics to be compared consistently across matches
and aggregated later for tournament-level analysis.


In [0]:
display(
    df_silver
    .filter(
        (col("event_type") == "Shot") &
        (col("shot_outcome") == "Goal")
    )
    .groupBy("team")
    .agg(
        count("*").alias("goals")
    )
    .orderBy(col("goals").desc())
)

team,goals
England,6
France,4
Iran,2
Saudi Arabia,2
Netherlands,2
Ecuador,2
Australia,1
United States,1
Belgium,1
Wales,1


In [0]:
display(
    df_silver
    .filter(
        (col("event_type") == "Shot") &
        (col("shot_outcome") == "Goal")
    )
    .agg(
        count("*").alias("total_goals")
    )
)

total_goals
23


## 3. Build team match statistics

We now aggregate the Silver event data at team-match level.

The analytical grain is:

One row = one team in one match.

The first set of metrics focuses on attacking and possession-related activity:
total events, passes, completed passes, shots, goals, and expected goals (xG).

The metrics are calculated directly from the event-level Silver dataset using
the definitions validated in the previous steps.


In [0]:
df_team_match = (
    df_silver
    .groupBy(
        "match_id",
        "team"
    )
    .agg(
        count("*").alias("total_events"),

        count(
            when(col("event_type") == "Pass", True)
        ).alias("passes"),

        count(
            when(
                (col("event_type") == "Pass") &
                (col("pass_outcome") == "Complete"),
                True
            )
        ).alias("completed_passes"),

        count(
            when(col("event_type") == "Shot", True)
        ).alias("shots"),

        count(
            when(
                (col("event_type") == "Shot") &
                (col("shot_outcome") == "Goal"),
                True
            )
        ).alias("goals"),

        sum(
            when(
                col("event_type") == "Shot",
                col("shot_xg")
            ).otherwise(0)
        ).alias("xg")
    )
)

display(df_team_match)

match_id,team,total_events,passes,completed_passes,shots,goals,xg
3857254,Denmark,2146,646,544,11,0,1.5665587579999998
3857254,Tunisia,1534,417,315,13,0,1.0599351179999998
3857265,Mexico,1822,544,437,11,0,0.5291349314
3857265,Poland,1362,362,259,6,0,1.137926228
3857268,Belgium,1809,534,451,9,1,0.6700408539999999
3857268,Canada,1690,480,401,22,0,2.5462123360000004
3857271,England,2619,846,746,13,6,1.987157745
3857271,Iran,1030,247,163,8,2,1.4556585987
3857277,Morocco,1498,384,301,8,0,0.36263897600000006
3857277,Croatia,2221,686,577,5,0,0.8498605050000001


## 4. Validate team-level aggregations

Before adding additional metrics, we validate that the aggregation from event level
to team-match level preserves the totals observed in the Silver dataset.

The aggregated totals should remain consistent with the previously validated
Silver data:

* 10,277 passes
* 200 shots
* 23 goals

This confirms that the team-match aggregation is not losing events or creating
unexpected duplicates.


In [0]:
display(
    df_team_match.agg(
        sum("passes").alias("total_passes"),
        sum("shots").alias("total_shots"),
        sum("goals").alias("total_goals")
    )
)

total_passes,total_shots,total_goals
10277,200,23


## 5. Calculate pass completion percentage

Pass completion percentage measures the proportion of attempted passes that
were classified as complete.

Formula:

pass_completion_pct = completed_passes / passes × 100

The calculation is performed at the team-match level.

A division-by-zero condition is handled explicitly to avoid invalid results
for any future dataset where a team may have no recorded passes.


In [0]:
from pyspark.sql.functions import when, round

df_team_match = (
    df_team_match
    .withColumn(
        "pass_completion_pct",
        when(
            col("passes") > 0,
            round(
                col("completed_passes") / col("passes") * 100,
                2
            )
        ).otherwise(0)
    )
)

display(df_team_match)

match_id,team,total_events,passes,completed_passes,shots,goals,xg,pass_completion_pct
3857254,Denmark,2146,646,544,11,0,1.5665587579999998,84.21
3857254,Tunisia,1534,417,315,13,0,1.0599351179999998,75.54
3857265,Mexico,1822,544,437,11,0,0.5291349314,80.33
3857265,Poland,1362,362,259,6,0,1.137926228,71.55
3857268,Belgium,1809,534,451,9,1,0.6700408539999999,84.46
3857268,Canada,1690,480,401,22,0,2.5462123360000004,83.54
3857271,England,2619,846,746,13,6,1.987157745,88.18
3857271,Iran,1030,247,163,8,2,1.4556585987,65.99
3857277,Morocco,1498,384,301,8,0,0.36263897600000006,78.39
3857277,Croatia,2221,686,577,5,0,0.8498605050000001,84.11


## 6. Add defensive and possession metrics

The team-match dataset will be extended with three additional event-based metrics:

* `pressures`: number of `Pressure` events performed by the team.
* `ball_recoveries`: number of `Ball Recovery` events performed by the team.
* `dispossessions`: number of `Dispossessed` events involving the team's players.

These metrics are calculated directly from the event types available in the
Silver layer.

The aggregation remains at the same grain:

One row = one team in one match.


In [0]:
from pyspark.sql.functions import col, count, countDistinct, sum, when, round

In [0]:


df_team_match = (
    df_silver
    .groupBy(
        "match_id",
        "team"
    )
    .agg(
        # General event metrics
        count("*").alias("total_events"),

        # Passing metrics
        count(
            when(col("event_type") == "Pass", True)
        ).alias("passes"),

        count(
            when(
                (col("event_type") == "Pass") &
                (col("pass_outcome") == "Complete"),
                True
            )
        ).alias("completed_passes"),

        # Shooting metrics
        count(
            when(col("event_type") == "Shot", True)
        ).alias("shots"),

        count(
            when(
                (col("event_type") == "Shot") &
                (col("shot_outcome") == "Goal"),
                True
            )
        ).alias("goals"),

        # Expected goals
        sum(
            when(
                col("event_type") == "Shot",
                col("shot_xg")
            ).otherwise(0)
        ).alias("xg"),

        # Defensive and possession metrics
        count(
            when(col("event_type") == "Pressure", True)
        ).alias("pressures"),

        count(
            when(col("event_type") == "Ball Recovery", True)
        ).alias("ball_recoveries"),

        count(
            when(col("event_type") == "Dispossessed", True)
        ).alias("dispossessions")
    )
    
    # Pass completion percentage
    .withColumn(
        "pass_completion_pct",
        when(
            col("passes") > 0,
            round(
                col("completed_passes") / col("passes") * 100,
                2
            )
        ).otherwise(0)
    )
)

display(df_team_match)

match_id,team,total_events,passes,completed_passes,shots,goals,xg,pressures,ball_recoveries,dispossessions,pass_completion_pct
3857254,Denmark,2146,646,544,11,0,1.5665587579999998,109,44,14,84.21
3857254,Tunisia,1534,417,315,13,0,1.0599351179999998,130,51,11,75.54
3857265,Mexico,1822,544,437,11,0,0.5291349314,96,49,13,80.33
3857265,Poland,1362,362,259,6,0,1.137926228,135,49,17,71.55
3857268,Belgium,1809,534,451,9,1,0.6700408539999999,83,39,8,84.46
3857268,Canada,1690,480,401,22,0,2.5462123360000004,112,50,9,83.54
3857271,England,2619,846,746,13,6,1.987157745,101,46,8,88.18
3857271,Iran,1030,247,163,8,2,1.4556585987,141,38,7,65.99
3857277,Morocco,1498,384,301,8,0,0.36263897600000006,168,55,11,78.39
3857277,Croatia,2221,686,577,5,0,0.8498605050000001,107,43,17,84.11


## 7. Validate defensive and possession metrics

Before persisting the Gold dataset, the defensive and possession metrics are
validated against the Silver event-level data.

The totals calculated independently from Silver must match the totals produced
by the Gold team-match aggregation.

This validation checks:

* `pressures`
* `ball_recoveries`
* `dispossessions`

Matching totals confirm that the aggregation logic is preserving the underlying
event data correctly.


In [0]:
display(
    df_silver.agg(
        count(
            when(col("event_type") == "Pressure", True)
        ).alias("total_pressures"),

        count(
            when(col("event_type") == "Ball Recovery", True)
        ).alias("total_ball_recoveries"),

        count(
            when(col("event_type") == "Dispossessed", True)
        ).alias("total_dispossessions")
    )
)

total_pressures,total_ball_recoveries,total_dispossessions
2332,892,220


In [0]:
display(
    df_team_match.agg(
        sum("pressures").alias("total_pressures"),
        sum("ball_recoveries").alias("total_ball_recoveries"),
        sum("dispossessions").alias("total_dispossessions")
    )
)

total_pressures,total_ball_recoveries,total_dispossessions
2332,892,220


## 8. Validate expected goals

Expected goals (xG) is calculated in the Gold layer by summing the
`shot_xg` value for all Shot events belonging to each team and match.

Because xG is a numeric measure rather than an event count, it requires a
separate aggregation validation.

The total xG calculated directly from the Silver event-level data must match
the total xG produced by the Gold team-match aggregation.

This confirms that no shot-level xG values were lost or duplicated during
the aggregation.


In [0]:
display(
    df_silver
    .filter(col("event_type") == "Shot")
    .agg(
        sum("shot_xg").alias("silver_total_xg")
    )
)

silver_total_xg
23.575419028099994


In [0]:
display(
    df_team_match.agg(
        sum("xg").alias("gold_total_xg")
    )
)

gold_total_xg
23.575419028099994


## 9. Validate Gold dataset structure

The team-match Gold dataset is now functionally complete.

Before persisting the dataset as a Delta table, structural quality checks are
performed to ensure that:

* Each match contains exactly two teams.
* The dataset contains the expected number of team-match records.
* Analytical metrics do not contain unexpected null values.
* The resulting schema is suitable for downstream analytics.

These checks complement the metric-level validations performed in the previous
sections.


In [0]:
display(
    df_team_match.agg(
        count("*").alias("total_team_matches"),
        countDistinct("match_id").alias("total_matches"),
        countDistinct("team").alias("total_teams")
    )
)

total_team_matches,total_matches,total_teams
20,10,20


## 10. Validate match-team cardinality

Each football match should contain exactly two teams.

This validation groups the Gold dataset by `match_id` and counts the number of
distinct teams associated with each match.

The expected result is exactly two teams for every match.

This check helps detect potential duplication or aggregation issues before the
dataset is persisted.


In [0]:
display(
    df_team_match
    .groupBy("match_id")
    .agg(
        countDistinct("team").alias("team_count")
    )
    .orderBy("match_id")
)

match_id,team_count
3857254,2
3857265,2
3857268,2
3857271,2
3857277,2
3857279,2
3857282,2
3857285,2
3857286,2
3857300,2


## 11. Validate Gold metric completeness

The final Gold dataset should not contain unexpected null values in its
analytical metrics.

The following columns are validated:

* `total_events`
* `passes`
* `completed_passes`
* `shots`
* `goals`
* `xg`
* `pressures`
* `ball_recoveries`
* `dispossessions`
* `pass_completion_pct`

Null values in these metrics could affect downstream aggregations and
visualizations, so they are checked before the dataset is persisted.


In [0]:
display(
    df_team_match.select(
        count(when(col("total_events").isNull(), True)).alias("null_total_events"),
        count(when(col("passes").isNull(), True)).alias("null_passes"),
        count(when(col("completed_passes").isNull(), True)).alias("null_completed_passes"),
        count(when(col("shots").isNull(), True)).alias("null_shots"),
        count(when(col("goals").isNull(), True)).alias("null_goals"),
        count(when(col("xg").isNull(), True)).alias("null_xg"),
        count(when(col("pressures").isNull(), True)).alias("null_pressures"),
        count(when(col("ball_recoveries").isNull(), True)).alias("null_ball_recoveries"),
        count(when(col("dispossessions").isNull(), True)).alias("null_dispossessions"),
        count(when(col("pass_completion_pct").isNull(), True)).alias("null_pass_completion_pct")
    )
)

null_total_events,null_passes,null_completed_passes,null_shots,null_goals,null_xg,null_pressures,null_ball_recoveries,null_dispossessions,null_pass_completion_pct
0,0,0,0,0,0,0,0,0,0


## 12. Review Gold schema

Before persisting the Gold dataset, the schema is reviewed to ensure that
identifiers, counters, percentages, and numerical measures use appropriate
data types.

This step helps prevent unnecessary type conversions in downstream analytical
tools such as Power BI and improves the consistency of the Gold data model.


In [0]:
df_team_match.printSchema()

root
 |-- match_id: long (nullable = true)
 |-- team: string (nullable = true)
 |-- total_events: long (nullable = false)
 |-- passes: long (nullable = false)
 |-- completed_passes: long (nullable = false)
 |-- shots: long (nullable = false)
 |-- goals: long (nullable = false)
 |-- xg: double (nullable = true)
 |-- pressures: long (nullable = false)
 |-- ball_recoveries: long (nullable = false)
 |-- dispossessions: long (nullable = false)
 |-- pass_completion_pct: double (nullable = true)



## 13. Persist team match analytics

The validated team-match dataset is now ready to be persisted in the Gold layer.

The table `golstats.gold.estadisticas_equipo` will store one record per team
and match, providing a curated analytical dataset for downstream reporting
and visualization.

The table is stored in Delta format and registered in Unity Catalog.

Gold grain:

One row = one team in one match.

This table is intended to serve as a reusable analytical source for tools such
as Power BI without requiring downstream consumers to repeat the event-level
transformations.


In [0]:
%sql
DROP TABLE IF EXISTS golstats.gold.estadisticas_equipo;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS golstats.gold;

In [0]:
GOLD_TABLE = "golstats.gold.estadisticas_equipo"

(
    df_team_match
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(GOLD_TABLE)
)

## 14. Validate persisted Gold table

The Gold dataset has been successfully persisted as a Delta table in Unity Catalog.

The table is now read back from the catalog to validate the persisted data
independently from the in-memory DataFrame.

This validation confirms:

- The expected number of team-match records was persisted.
- All expected matches are present.
- All expected teams are present.
- Core analytical metrics were preserved.
- The persisted table can be consumed independently by downstream tools.

In [0]:
from pyspark.sql.functions import count, countDistinct, sum

df_gold = spark.table("golstats.gold.estadisticas_equipo")

display(df_gold)

match_id,team,total_events,passes,completed_passes,shots,goals,xg,pressures,ball_recoveries,dispossessions,pass_completion_pct
3857286,Qatar,1582,468,382,5,0,0.30418895050000005,99,33,8,81.62
3857300,Saudi Arabia,1201,286,197,3,2,0.153461642,141,49,4,68.88
3857282,United States,2096,610,520,6,1,0.6024987946,137,45,22,85.25
3857277,Croatia,2221,686,577,5,0,0.8498605050000001,107,43,17,84.11
3857271,Iran,1030,247,163,8,2,1.4556585987,141,38,7,65.99
3857285,Netherlands,1599,491,393,10,2,1.1238838,104,31,9,80.04
3857265,Poland,1362,362,259,6,0,1.137926228,135,49,17,71.55
3857282,Wales,1591,443,339,7,1,1.395336012,128,53,9,76.52
3857268,Belgium,1809,534,451,9,1,0.6700408539999999,83,39,8,84.46
3857277,Morocco,1498,384,301,8,0,0.36263897600000006,168,55,11,78.39


In [0]:
display(
    df_gold.agg(
        count("*").alias("total_rows"),
        countDistinct("match_id").alias("total_matches"),
        countDistinct("team").alias("total_teams"),
        sum("passes").alias("total_passes"),
        sum("shots").alias("total_shots"),
        sum("goals").alias("total_goals"),
        sum("xg").alias("total_xg")
    )
)

total_rows,total_matches,total_teams,total_passes,total_shots,total_goals,total_xg
20,10,20,10277,200,23,23.5754190281


## 15. Gold dataset summary

The `golstats.gold.estadisticas_equipo` table provides team-level football
performance metrics at match level.

### Dataset grain

One row represents one team in one match.

### Analytical metrics

The dataset contains:

* `total_events`: total number of recorded events for the team.
* `passes`: total pass attempts.
* `completed_passes`: passes classified as complete.
* `pass_completion_pct`: percentage of completed passes.
* `shots`: total shots.
* `goals`: goals scored.
* `xg`: total expected goals.
* `pressures`: defensive pressure events.
* `ball_recoveries`: ball recovery events.
* `dispossessions`: times the team was dispossessed.

### Data quality

The persisted Gold table was validated against the Silver event-level data.

Validation results:

* 20 team-match records.
* 10 matches.
* 20 teams.
* 10,277 passes.
* 200 shots.
* 23 goals.
* 23.5754 total xG.

The successful validation confirms that the Gold aggregation preserves the
underlying event-level information and is ready for downstream analytics.

The table can now be consumed by analytical tools such as Power BI without
requiring the event-level transformations to be repeated.


## 16. Investigate player-level data

Before creating the player-level Gold dataset, the Silver event data is
investigated to understand how player information is represented across
different event types.

The objective is to determine:

* Which event types contain player information.
* How many unique players are present in the dataset.
* Whether player identifiers are consistently available.
* Which event types should contribute to player-level performance metrics.

This investigation is performed before aggregation to ensure that the Gold
business logic is based on the actual structure of the source data.


In [0]:
display(
    df_silver
    .groupBy("event_type")
    .agg(
        count("*").alias("total_events"),
        count("player_id").alias("events_with_player"),
        countDistinct("player_id").alias("unique_players")
    )
    .orderBy(col("total_events").desc())
)

event_type,total_events,events_with_player,unique_players
Pass,10277,10277,303
Ball Receipt*,9514,9514,301
Carry,8171,8171,297
Pressure,2332,2332,277
Ball Recovery,892,892,258
Duel,640,640,228
Clearance,408,408,152
Block,380,380,192
Miscontrol,277,277,154
Foul Committed,272,272,159


## 17. Define player performance metrics

The player-level Gold dataset will aggregate event-level data into match-level
player performance metrics.

The grain of the dataset will be:

**One row = one player in one match.**

Only events containing a valid player identifier will contribute to the
player-level dataset.

The initial metrics will include:

* `total_events`: total number of recorded events involving the player.
* `passes`: total pass attempts.
* `completed_passes`: passes classified as complete.
* `shots`: total shots attempted.
* `goals`: shots with a `Goal` outcome.
* `xg`: total expected goals generated from the player's shots.
* `pressures`: number of pressure events.
* `ball_recoveries`: number of ball recovery events.
* `dispossessions`: number of times the player was dispossessed.

Player identity will be represented using both `player_id` and `player` to provide
a stable identifier and a human-readable name.

The `position` field will also be retained as contextual information for
downstream analysis.

Events without a player identifier will not be included in the player-level
aggregation because they cannot be attributed to an individual player.


In [0]:
df_silver_player_events = (
    df_silver
    .filter(col("player_id").isNotNull())
)

In [0]:

df_player_match = (
    df_silver_player_events
    .groupBy(
        "match_id",
        "team",
        "player_id",
        "player",
        "position"
    )
    .agg(
        # General event metrics
        count("*").alias("total_events"),

        # Passing metrics
        count(
            when(col("event_type") == "Pass", True)
        ).alias("passes"),

        count(
            when(
                (col("event_type") == "Pass") &
                (col("pass_outcome") == "Complete"),
                True
            )
        ).alias("completed_passes"),

        # Shooting metrics
        count(
            when(col("event_type") == "Shot", True)
        ).alias("shots"),

        count(
            when(
                (col("event_type") == "Shot") &
                (col("shot_outcome") == "Goal"),
                True
            )
        ).alias("goals"),

        # Expected goals
        sum(
            when(
                col("event_type") == "Shot",
                col("shot_xg")
            ).otherwise(0)
        ).alias("xg"),

        # Defensive and possession metrics
        count(
            when(col("event_type") == "Pressure", True)
        ).alias("pressures"),

        count(
            when(col("event_type") == "Ball Recovery", True)
        ).alias("ball_recoveries"),

        count(
            when(col("event_type") == "Dispossessed", True)
        ).alias("dispossessions")
    )
)

display(df_player_match)

match_id,team,player_id,player,position,total_events,passes,completed_passes,shots,goals,xg,pressures,ball_recoveries,dispossessions
3857254,Denmark,6302,Kasper Dolberg,Left Center Forward,66,10,9,1,0,0.124032564,14,2,0
3857254,Denmark,3043,Christian Dannemann Eriksen,Left Center Midfield,149,46,38,0,0,0.0,5,2,2
3857254,Denmark,17042,Andreas Skov Olsen,Right Center Forward,85,22,14,0,0,0.0,7,0,3
3857254,Tunisia,23910,Youssef Msakni,Left Wing,120,25,19,3,0,0.139569514,9,4,3
3857254,Denmark,16190,Rasmus Nissen Kristensen,Right Wing Back,145,49,36,0,0,0.0,12,1,0
3857254,Tunisia,5651,Yassine Meriah,Center Back,171,53,44,1,0,0.046073906,9,1,0
3857254,Denmark,16554,Joakim Mæhle,Left Wing Back,122,38,32,0,0,0.0,6,2,0
3857254,Tunisia,30681,Ali Abdi,Left Wing Back,147,44,27,0,0,0.0,15,4,3
3857254,Tunisia,35592,Anis Ben Slimane,Right Wing,95,24,17,0,0,0.0,11,2,2
3857254,Tunisia,9236,Mohamed Dräger,Right Wing Back,114,31,23,2,0,0.0540321,14,2,1


## 17.1 Correct player aggregation grain

During the player-level aggregation, the investigation showed that a player's
`position` can change during the same match.

StatsBomb records the player's position at the event level, meaning that the
same player can appear with multiple positions within a single match.

Therefore, `position` cannot be part of the primary aggregation grain.

The player performance table will use the following grain:

**One row = one player in one match.**

The aggregation key will therefore be:

* `match_id`
* `team`
* `player_id`
* `player`

The `position` field will not be included in the primary player performance
table because doing so would create multiple rows for the same player and
match.

Position information can be preserved separately for future tactical analysis
or represented as a derived attribute if required.


In [0]:

df_player_match = (
    df_silver_player_events
    .groupBy(
        "match_id",
        "team",
        "player_id",
        "player"
    )
    .agg(
        # General event metrics
        count("*").alias("total_events"),

        # Passing metrics
        count(
            when(col("event_type") == "Pass", True)
        ).alias("passes"),

        count(
            when(
                (col("event_type") == "Pass") &
                (col("pass_outcome") == "Complete"),
                True
            )
        ).alias("completed_passes"),

        # Shooting metrics
        count(
            when(col("event_type") == "Shot", True)
        ).alias("shots"),

        count(
            when(
                (col("event_type") == "Shot") &
                (col("shot_outcome") == "Goal"),
                True
            )
        ).alias("goals"),

        # Expected goals
        sum(
            when(
                col("event_type") == "Shot",
                col("shot_xg")
            ).otherwise(0)
        ).alias("xg"),

        # Defensive and possession metrics
        count(
            when(col("event_type") == "Pressure", True)
        ).alias("pressures"),

        count(
            when(col("event_type") == "Ball Recovery", True)
        ).alias("ball_recoveries"),

        count(
            when(col("event_type") == "Dispossessed", True)
        ).alias("dispossessions")
    )
)

display(df_player_match)

match_id,team,player_id,player,total_events,passes,completed_passes,shots,goals,xg,pressures,ball_recoveries,dispossessions
3857254,Denmark,6302,Kasper Dolberg,77,11,10,2,0,0.380551344,15,2,1
3857254,Denmark,3043,Christian Dannemann Eriksen,254,80,66,1,0,0.08509644,10,4,2
3857254,Denmark,17042,Andreas Skov Olsen,111,28,19,0,0,0.0,9,0,3
3857254,Tunisia,23910,Youssef Msakni,120,25,19,3,0,0.139569514,9,4,3
3857254,Denmark,16190,Rasmus Nissen Kristensen,171,58,44,0,0,0.0,12,1,0
3857254,Tunisia,5651,Yassine Meriah,171,53,44,1,0,0.046073906,9,1,0
3857254,Denmark,16554,Joakim Mæhle,170,52,43,1,0,0.028684001,7,3,1
3857254,Tunisia,30681,Ali Abdi,147,44,27,0,0,0.0,15,4,3
3857254,Tunisia,35592,Anis Ben Slimane,95,24,17,0,0,0.0,11,2,2
3857254,Tunisia,9236,Mohamed Dräger,114,31,23,2,0,0.0540321,14,2,1


## 18. Validate player-match grain

The player-level dataset must contain exactly one record for each player and
match combination.

This validation checks whether any player appears more than once within the
same match and team.

The expected result is zero duplicated player-match combinations.


In [0]:
df_player_duplicates = (
    df_player_match
    .groupBy(
        "match_id",
        "team",
        "player_id"
    )
    .agg(
        count("*").alias("records")
    )
    .filter(col("records") > 1)
)

display(df_player_duplicates)

match_id,team,player_id,records


## 19. Validate player-level metrics

After validating the player-match grain, the next step is to verify that the
player-level aggregation preserves the underlying event data.

The totals generated by the player-match dataset will be compared against the
Silver event-level dataset.

The following metrics are validated:

* Passes
* Shots
* Goals
* Expected goals (xG)
* Pressures
* Ball recoveries
* Dispossessions

The totals should match the corresponding Silver totals exactly.


In [0]:
display(
    df_player_match.agg(
        sum("passes").alias("total_passes"),
        sum("shots").alias("total_shots"),
        sum("goals").alias("total_goals"),
        sum("xg").alias("total_xg"),
        sum("pressures").alias("total_pressures"),
        sum("ball_recoveries").alias("total_ball_recoveries"),
        sum("dispossessions").alias("total_dispossessions")
    )
)

total_passes,total_shots,total_goals,total_xg,total_pressures,total_ball_recoveries,total_dispossessions
10277,200,23,23.575419028099994,2332,892,220


## 20. Calculate player pass completion percentage

Pass completion percentage measures the proportion of attempted passes that were
classified as complete for each player in each match.

Formula:

pass_completion_pct = completed_passes / passes × 100

The calculation is performed at the player-match level.

A division-by-zero condition is handled explicitly to prevent invalid results
for players without recorded passes.


In [0]:
df_player_match = (
    df_player_match
    .withColumn(
        "pass_completion_pct",
        when(
            col("passes") > 0,
            round(
                col("completed_passes") / col("passes") * 100,
                2
            )
        ).otherwise(0)
    )
)

display(df_player_match)

match_id,team,player_id,player,total_events,passes,completed_passes,shots,goals,xg,pressures,ball_recoveries,dispossessions,pass_completion_pct
3857254,Denmark,6302,Kasper Dolberg,77,11,10,2,0,0.380551344,15,2,1,90.91
3857254,Denmark,3043,Christian Dannemann Eriksen,254,80,66,1,0,0.08509644,10,4,2,82.5
3857254,Denmark,17042,Andreas Skov Olsen,111,28,19,0,0,0.0,9,0,3,67.86
3857254,Tunisia,23910,Youssef Msakni,120,25,19,3,0,0.139569514,9,4,3,76.0
3857254,Denmark,16190,Rasmus Nissen Kristensen,171,58,44,0,0,0.0,12,1,0,75.86
3857254,Tunisia,5651,Yassine Meriah,171,53,44,1,0,0.046073906,9,1,0,83.02
3857254,Denmark,16554,Joakim Mæhle,170,52,43,1,0,0.028684001,7,3,1,82.69
3857254,Tunisia,30681,Ali Abdi,147,44,27,0,0,0.0,15,4,3,61.36
3857254,Tunisia,35592,Anis Ben Slimane,95,24,17,0,0,0.0,11,2,2,70.83
3857254,Tunisia,9236,Mohamed Dräger,114,31,23,2,0,0.0540321,14,2,1,74.19


## 20.1 Validate pass completion percentage

The calculated pass completion percentage is validated against the underlying
`passes` and `completed_passes` metrics.

The validation checks that:

* Players with recorded passes have a percentage between 0 and 100.
* Players without recorded passes have a percentage of 0.
* No invalid percentage values are present.

This ensures that the derived KPI is mathematically consistent before the
dataset is persisted.


In [0]:
display(
    df_player_match.agg(
        count(
            when(
                (col("pass_completion_pct") < 0) |
                (col("pass_completion_pct") > 100),
                True
            )
        ).alias("invalid_percentages"),

        count(
            when(
                (col("passes") == 0) &
                (col("pass_completion_pct") != 0),
                True
            )
        ).alias("zero_passes_with_invalid_percentage")
    )
)

invalid_percentages,zero_passes_with_invalid_percentage
0,0


## 21. Validate NULL values in player metrics

Before persisting the player-level Gold dataset, the analytical metrics are
validated for unexpected NULL values.

The following fields are considered essential for the player-match dataset:

* `match_id`
* `team`
* `player_id`
* `player`
* `total_events`
* `passes`
* `completed_passes`
* `shots`
* `goals`
* `xg`
* `pressures`
* `ball_recoveries`
* `dispossessions`
* `pass_completion_pct`

Unexpected NULL values in these fields could affect downstream aggregations
and analytical visualizations.


In [0]:
display(
    df_player_match.select(
        count(when(col("match_id").isNull(), True)).alias("null_match_id"),
        count(when(col("team").isNull(), True)).alias("null_team"),
        count(when(col("player_id").isNull(), True)).alias("null_player_id"),
        count(when(col("player").isNull(), True)).alias("null_player"),
        count(when(col("total_events").isNull(), True)).alias("null_total_events"),
        count(when(col("passes").isNull(), True)).alias("null_passes"),
        count(when(col("completed_passes").isNull(), True)).alias("null_completed_passes"),
        count(when(col("shots").isNull(), True)).alias("null_shots"),
        count(when(col("goals").isNull(), True)).alias("null_goals"),
        count(when(col("xg").isNull(), True)).alias("null_xg"),
        count(when(col("pressures").isNull(), True)).alias("null_pressures"),
        count(when(col("ball_recoveries").isNull(), True)).alias("null_ball_recoveries"),
        count(when(col("dispossessions").isNull(), True)).alias("null_dispossessions"),
        count(when(col("pass_completion_pct").isNull(), True)).alias("null_pass_completion_pct")
    )
)

null_match_id,null_team,null_player_id,null_player,null_total_events,null_passes,null_completed_passes,null_shots,null_goals,null_xg,null_pressures,null_ball_recoveries,null_dispossessions,null_pass_completion_pct
0,0,0,0,0,0,0,0,0,0,0,0,0,0


## 22. Review player Gold schema

Before persisting the player-level Gold dataset, the schema is reviewed to
ensure that identifiers, counters, percentages, and numerical measures use
appropriate data types.

The schema should be suitable for downstream analytical tools such as Power BI
and should not require unnecessary type conversions.


In [0]:
df_player_match.printSchema()

root
 |-- match_id: long (nullable = true)
 |-- team: string (nullable = true)
 |-- player_id: long (nullable = true)
 |-- player: string (nullable = true)
 |-- total_events: long (nullable = false)
 |-- passes: long (nullable = false)
 |-- completed_passes: long (nullable = false)
 |-- shots: long (nullable = false)
 |-- goals: long (nullable = false)
 |-- xg: double (nullable = true)
 |-- pressures: long (nullable = false)
 |-- ball_recoveries: long (nullable = false)
 |-- dispossessions: long (nullable = false)
 |-- pass_completion_pct: double (nullable = true)



## 23. Persist player-level Gold dataset

The player-level analytical dataset has passed the required quality checks.

The dataset will now be persisted as a Delta table in the Gold layer.

Table:

`golstats.gold.estadisticas_jugador`

The table grain is:

**One row = one player in one match.**

The dataset is designed to support player-level analysis in downstream
analytical tools such as Power BI.


In [0]:
GOLD_PLAYER_TABLE = "golstats.gold.estadisticas_jugador"

(
    df_player_match
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(GOLD_PLAYER_TABLE)
)

## 24 Create match-level Gold dataset

Before creating the match-level Gold dataset, the available match metadata is
reviewed to determine which fields can be reliably used in the analytical model.

The objective is to identify:

* Match date
* Home team
* Away team
* Home team score
* Away team score
* Match result

The investigation will determine whether these attributes are already available
in the ingested match metadata or need to be derived from the event-level data.

This step is performed before the Gold transformation to ensure that the match
dataset is based on validated source attributes.


The match-level Gold dataset requires reliable match metadata such as the match
date, participating teams, and final score.

The original ingestion notebook created an in-memory DataFrame containing part
of this information, but variables from another notebook are not available in
the current execution context.

Therefore, the Gold layer should rely on persisted data or explicitly reload
the required source data rather than depending on variables created in previous
notebooks.

In this step, the available persisted match metadata is inspected before
building the match-level analytical dataset.



In [0]:
import json

MATCHES_METADATA_PATH = "/Volumes/golstats/bronze/raw_files/matches_metadata.json"

with open(MATCHES_METADATA_PATH, "r") as f:
    matches_metadata = json.load(f)

display(matches_metadata)

away_team,home_team,match_date,match_id
Ecuador,Qatar,2022-11-20,3857286
Iran,England,2022-11-21,3857271
Wales,United States,2022-11-21,3857282
Netherlands,Senegal,2022-11-21,3857285
Tunisia,Denmark,2022-11-22,3857254
Poland,Mexico,2022-11-22,3857265
Australia,France,2022-11-22,3857279
Saudi Arabia,Argentina,2022-11-22,3857300
Canada,Belgium,2022-11-23,3857268
Croatia,Morocco,2022-11-23,3857277


## 24.2 Reload official match metadata

The persisted match metadata contains the match identifiers, dates, and
participating teams, but it does not include the final score.

Since the final score is an important analytical attribute, it will be loaded
from the official StatsBomb competition match metadata rather than inferred
from event records.

This preserves the source-of-truth principle and avoids deriving a match result
from downstream event data when the official match result is available directly
from the source.

The resulting dataset will contain:

* `match_id`
* `match_date`
* `home_team`
* `away_team`
* `home_score`
* `away_score`


In [0]:
import requests

BASE_URL = "https://raw.githubusercontent.com/statsbomb/open-data/master/data"

COMPETITION_ID = 43
SEASON_ID = 106

matches_url = f"{BASE_URL}/matches/{COMPETITION_ID}/{SEASON_ID}.json"

response = requests.get(matches_url)
response.raise_for_status()

matches_full = response.json()

print(f"Total matches retrieved: {len(matches_full)}")

Total matches retrieved: 64


In [0]:
display(matches_full[0])

{'match_id': 3857276,
 'match_date': '2022-12-01',
 'kick_off': '15:00:00.000',
 'competition': {'competition_id': 43,
  'country_name': 'International',
  'competition_name': 'FIFA World Cup'},
 'season': {'season_id': 106, 'season_name': '2022'},
 'home_team': {'home_team_id': 1833,
  'home_team_name': 'Canada',
  'home_team_gender': 'male',
  'home_team_group': 'F',
  'country': {'id': 40, 'name': 'Canada'},
  'managers': [{'id': 4435,
    'name': 'John Herdman',
    'nickname': None,
    'dob': '1975-07-19',
    'country': {'id': 68, 'name': 'England'}}]},
 'away_team': {'away_team_id': 788,
  'away_team_name': 'Morocco',
  'away_team_gender': 'male',
  'away_team_group': None,
  'country': {'id': 154, 'name': 'Morocco'},
  'managers': [{'id': 1000518,
    'name': 'Walid Regragui',
    'nickname': None,
    'dob': '1975-09-23',
    'country': {'id': 154, 'name': 'Morocco'}}]},
 'home_score': 1,
 'away_score': 2,
 'match_status': 'available',
 'match_status_360': 'available',
 'last

## 24.3 Build match-level Gold dataset

The match-level Gold dataset will be built from the official StatsBomb match
metadata.

Only the matches selected during the ingestion phase will be included in this
initial analytical dataset.

The following attributes will be retained:

* `match_id`: unique match identifier.
* `match_date`: date of the match.
* `home_team`: home team name.
* `away_team`: away team name.
* `home_score`: goals scored by the home team.
* `away_score`: goals scored by the away team.

The final match result will be derived from the official home and away scores.

The resulting grain will be:

**One row = one match.**


In [0]:
import json

MATCHES_METADATA_PATH = "/Volumes/golstats/bronze/raw_files/matches_metadata.json"

with open(MATCHES_METADATA_PATH, "r") as f:
    matches_metadata = json.load(f)

selected_match_ids = [
    match["match_id"]
    for match in matches_metadata
]

print(f"Selected matches: {len(selected_match_ids)}")
print(selected_match_ids)

Selected matches: 10
[3857286, 3857271, 3857282, 3857285, 3857254, 3857265, 3857279, 3857300, 3857268, 3857277]


## 24.4 Extract match attributes

The official StatsBomb match metadata contains many nested attributes that are
not required for the current analytical model.

Only the fields necessary for match-level analysis will be extracted.

The transformation also derives the match result from the official final score:

* `Home` when the home team scored more goals.
* `Away` when the away team scored more goals.
* `Draw` when both teams scored the same number of goals.


In [0]:

matches_selected = [
    match
    for match in matches_full
    if match["match_id"] in selected_match_ids
]

df_matches_gold = spark.createDataFrame([
    {
        "match_id": match["match_id"],
        "match_date": match["match_date"],
        "home_team": match["home_team"]["home_team_name"],
        "away_team": match["away_team"]["away_team_name"],
        "home_score": match["home_score"],
        "away_score": match["away_score"]
    }
    for match in matches_selected
])

df_matches_gold = (
    df_matches_gold
    .withColumn(
        "result",
        when(
            col("home_score") > col("away_score"),
            "Home"
        )
        .when(
            col("away_score") > col("home_score"),
            "Away"
        )
        .otherwise("Draw")
    )
)

display(
    df_matches_gold
    .orderBy("match_date", "match_id")
)

away_score,away_team,home_score,home_team,match_date,match_id,result
2,Ecuador,0,Qatar,2022-11-20,3857286,Away
2,Iran,6,England,2022-11-21,3857271,Home
1,Wales,1,United States,2022-11-21,3857282,Draw
2,Netherlands,0,Senegal,2022-11-21,3857285,Away
0,Tunisia,0,Denmark,2022-11-22,3857254,Draw
0,Poland,0,Mexico,2022-11-22,3857265,Draw
1,Australia,4,France,2022-11-22,3857279,Home
2,Saudi Arabia,1,Argentina,2022-11-22,3857300,Away
0,Canada,1,Belgium,2022-11-23,3857268,Home
0,Croatia,0,Morocco,2022-11-23,3857277,Draw


## 24.5 Validate match-level grain

The match-level dataset must contain exactly one record for each match.

This validation checks:

* Total number of matches.
* Number of unique match identifiers.
* Duplicate match identifiers.
* Unexpected NULL values in critical fields.

The expected result is:

* 10 total records.
* 10 unique matches.
* 0 duplicated match identifiers.
* 0 NULL values in required fields.

This validation confirms that the dataset follows the intended grain:

**One row = one match.**


In [0]:
display(
    df_matches_gold.agg(
        count("*").alias("total_matches"),
        countDistinct("match_id").alias("unique_match_ids")
    )
)

total_matches,unique_match_ids
10,10


In [0]:
display(
    df_matches_gold
    .groupBy("match_id")
    .agg(
        count("*").alias("records")
    )
    .filter(col("records") > 1)
)

match_id,records


In [0]:
display(
    df_matches_gold.select(
        count(when(col("match_id").isNull(), True)).alias("null_match_id"),
        count(when(col("match_date").isNull(), True)).alias("null_match_date"),
        count(when(col("home_team").isNull(), True)).alias("null_home_team"),
        count(when(col("away_team").isNull(), True)).alias("null_away_team"),
        count(when(col("home_score").isNull(), True)).alias("null_home_score"),
        count(when(col("away_score").isNull(), True)).alias("null_away_score"),
        count(when(col("result").isNull(), True)).alias("null_result")
    )
)

null_match_id,null_match_date,null_home_team,null_away_team,null_home_score,null_away_score,null_result
0,0,0,0,0,0,0


## 25. Validate consistency between Gold datasets

The match-level and team-level Gold datasets are derived from different source
structures:

* `gold.partidos` contains the official final score from StatsBomb match metadata.
* `gold.estadisticas_equipo` contains goals derived from Shot events in the
  event-level dataset.

These datasets should be consistent.

For every match:

* The home team's `goals` must equal `home_score`.
* The away team's `goals` must equal `away_score`.
* The sum of team goals must equal the total goals recorded in the match.

This cross-dataset validation ensures that independently derived Gold datasets
represent the same underlying football events consistently.


In [0]:
df_matches_gold = spark.table("golstats.gold.partidos") \
    if spark.catalog.tableExists("golstats.gold.partidos") \
    else df_matches_gold

df_team_gold = spark.table("golstats.gold.estadisticas_equipo")

display(df_team_gold)

match_id,team,total_events,passes,completed_passes,shots,goals,xg,pressures,ball_recoveries,dispossessions,pass_completion_pct
3857286,Qatar,1582,468,382,5,0,0.30418895050000005,99,33,8,81.62
3857300,Saudi Arabia,1201,286,197,3,2,0.153461642,141,49,4,68.88
3857282,United States,2096,610,520,6,1,0.6024987946,137,45,22,85.25
3857277,Croatia,2221,686,577,5,0,0.8498605050000001,107,43,17,84.11
3857271,Iran,1030,247,163,8,2,1.4556585987,141,38,7,65.99
3857285,Netherlands,1599,491,393,10,2,1.1238838,104,31,9,80.04
3857265,Poland,1362,362,259,6,0,1.137926228,135,49,17,71.55
3857282,Wales,1591,443,339,7,1,1.395336012,128,53,9,76.52
3857268,Belgium,1809,534,451,9,1,0.6700408539999999,83,39,8,84.46
3857277,Morocco,1498,384,301,8,0,0.36263897600000006,168,55,11,78.39


## 25.2 Prepare cross-dataset goal validation

The team-level Gold dataset contains one row per team and match.

To compare it with the match-level dataset, the two team records will be
combined into a single match-level representation containing the goals scored
by each participating team.

The resulting dataset will then be compared against the official home and away
scores.


In [0]:
df_goal_validation = (
    df_matches_gold
    .join(
        df_team_gold,
        on="match_id",
        how="inner"
    )
    .groupBy(
        "match_id",
        "home_team",
        "away_team",
        "home_score",
        "away_score"
    )
    .agg(
        sum(
            when(
                col("team") == col("home_team"),
                col("goals")
            ).otherwise(0)
        ).alias("calculated_home_goals"),

        sum(
            when(
                col("team") == col("away_team"),
                col("goals")
            ).otherwise(0)
        ).alias("calculated_away_goals")
    )
)

display(
    df_goal_validation
    .orderBy("match_id")
)

match_id,home_team,away_team,home_score,away_score,calculated_home_goals,calculated_away_goals
3857254,Denmark,Tunisia,0,0,0,0
3857265,Mexico,Poland,0,0,0,0
3857268,Belgium,Canada,1,0,1,0
3857271,England,Iran,6,2,6,2
3857277,Morocco,Croatia,0,0,0,0
3857279,France,Australia,4,1,4,1
3857282,United States,Wales,1,1,1,1
3857285,Senegal,Netherlands,0,2,0,2
3857286,Qatar,Ecuador,0,2,0,2
3857300,Argentina,Saudi Arabia,1,2,1,2


## 25.3 Detect goal inconsistencies

The official match scores are compared against the goals calculated from the
event-level Shot data.

Any row returned by this validation represents a discrepancy between the
official match metadata and the analytical team statistics.

The expected result is zero inconsistent matches.


In [0]:
display(
    df_goal_validation
    .filter(
        (col("home_score") != col("calculated_home_goals")) |
        (col("away_score") != col("calculated_away_goals"))
    )
)

match_id,home_team,away_team,home_score,away_score,calculated_home_goals,calculated_away_goals


## 26. Persist match-level Gold dataset

The match-level dataset has successfully passed structural and cross-dataset
validation.

The official match scores are consistent with the goals calculated from the
Silver event data and stored in `gold.estadisticas_equipo`.

The validated dataset will now be persisted as a Delta table:

`golstats.gold.partidos`

The table grain is:

**One row = one match.**


In [0]:
GOLD_MATCH_TABLE = "golstats.gold.partidos"

(
    df_matches_gold
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(GOLD_MATCH_TABLE)
)

## 27. Validate persisted match Gold table

The persisted match-level Gold table is reloaded from Unity Catalog to verify
that the Delta table contains the expected records after persistence.

The validation checks:

* Total number of matches.
* Number of unique match IDs.
* Total home goals.
* Total away goals.
* Number of draws.
* Number of home wins.
* Number of away wins.

The persisted dataset should contain exactly the same analytical information
as the validated in-memory DataFrame.


In [0]:
df_gold_matches = spark.table("golstats.gold.partidos")

display(df_gold_matches)

away_score,away_team,home_score,home_team,match_date,match_id,result
2,Iran,6,England,2022-11-21,3857271,Home
0,Croatia,0,Morocco,2022-11-23,3857277,Draw
2,Netherlands,0,Senegal,2022-11-21,3857285,Away
1,Wales,1,United States,2022-11-21,3857282,Draw
0,Tunisia,0,Denmark,2022-11-22,3857254,Draw
0,Poland,0,Mexico,2022-11-22,3857265,Draw
0,Canada,1,Belgium,2022-11-23,3857268,Home
2,Ecuador,0,Qatar,2022-11-20,3857286,Away
2,Saudi Arabia,1,Argentina,2022-11-22,3857300,Away
1,Australia,4,France,2022-11-22,3857279,Home


In [0]:
display(
    df_gold_matches.agg(
        count("*").alias("total_matches"),
        countDistinct("match_id").alias("unique_match_ids"),
        sum("home_score").alias("total_home_goals"),
        sum("away_score").alias("total_away_goals"),
        count(
            when(col("result") == "Home", True)
        ).alias("home_wins"),
        count(
            when(col("result") == "Away", True)
        ).alias("away_wins"),
        count(
            when(col("result") == "Draw", True)
        ).alias("draws")
    )
)

total_matches,unique_match_ids,total_home_goals,total_away_goals,home_wins,away_wins,draws
10,10,13,10,3,3,4


## 28. Review Gold analytical model

The Gold layer now contains three analytical datasets with different levels of
granularity.

These datasets are designed to support different analytical use cases:

* `gold.partidos` provides match-level information.
* `gold.estadisticas_equipo` provides team performance for each match.
* `gold.estadisticas_jugador` provides player performance for each match.

Before moving to downstream analytics, the Gold layer will be reviewed to
confirm that each dataset follows its intended grain and that `match_id` can
be used consistently to relate the datasets.

The expected grains are:

* `partidos`: one row per match.
* `estadisticas_equipo`: one row per team per match.
* `estadisticas_jugador`: one row per player per match.

This establishes the analytical foundation that will later be consumed by
Power BI and Machine Learning workflows.


28.1 Validate Gold table grains

In [0]:
df_gold_matches = spark.table("golstats.gold.partidos")

display(
    df_gold_matches.agg(
        count("*").alias("total_rows"),
        countDistinct("match_id").alias("unique_matches")
    )
)

total_rows,unique_matches
10,10


In [0]:
df_gold_teams = spark.table("golstats.gold.estadisticas_equipo")

display(
    df_gold_teams.agg(
        count("*").alias("total_rows"),
        countDistinct("match_id").alias("unique_matches"),
        countDistinct("team").alias("unique_teams")
    )
)

total_rows,unique_matches,unique_teams
20,10,20


In [0]:
display(
    df_gold_teams
    .groupBy("match_id")
    .agg(
        count("*").alias("teams_per_match")
    )
    .filter(col("teams_per_match") != 2)
)

match_id,teams_per_match


In [0]:
df_gold_players = spark.table("golstats.gold.estadisticas_jugador")

display(
    df_gold_players.agg(
        count("*").alias("total_rows"),
        countDistinct("match_id").alias("unique_matches"),
        countDistinct("player_id").alias("unique_players")
    )
)

total_rows,unique_matches,unique_players
305,10,305


## 30. Gold layer completion

The Gold layer has been successfully implemented and validated.

Three analytical datasets are now available:

* `golstats.gold.partidos`
* `golstats.gold.estadisticas_equipo`
* `golstats.gold.estadisticas_jugador`

Each dataset follows a clearly defined analytical grain:

* `partidos`: one row per match.
* `estadisticas_equipo`: one row per team per match.
* `estadisticas_jugador`: one row per player per match.

The datasets have been validated for structural integrity, metric consistency,
NULL values, duplicate records, and cross-dataset consistency.

The Gold layer is therefore ready to support downstream analytical workloads.

The current implementation uses a controlled subset of the FIFA World Cup 2022
matches for development and validation purposes.

The next phase will expand the ingestion scope to the complete competition
dataset while preserving the same Bronze, Silver, and Gold architecture.
